In [1]:
# Bootstrap values extraction using dabest
# Input: 2025collection OSAR data
# Output: CSV files with bootstraps, effect sizes, and CIs

import os
import numpy as np
import pandas as pd
import dabest
import warnings

warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 20.38it/s]

Numba compilation complete!


In [2]:
# Configuration
INPUT_DIR = r"D:\ACC Lab Dropbox\ACC Lab\Nicole Lee\Data Compilation\osar_compiled\2025collection"
OUTPUT_DIR = r"D:\ACC Lab Dropbox\ACC Lab\Nicole Lee\Data Compilation\osar_compiled\Bootstrapped stats"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Metrics to extract
PARAMETERS = [
    "pi_smoothed_Pattern 01", 
    "log2_speed_ratio_Pattern 01", 
    "log2_pace_ratio_Pattern 01",
    "light_attraction_index_Pattern 01", 
    "bout_index_Pattern 01", 
    "bout_duration_ratio_Pattern 01",
    "max_velocity_ratio_Pattern 01", 
    "speed_ratio_Pattern 01", 
    "pace_ratio_Pattern 01"
]

PARAMETERS_RENAME = [
    "PI_Hg_OSAR", 
    "Log2 Speed ratio_Hg_OSAR", 
    "Log2 Bout Speed ratio_Hg_OSAR",
    "Light Attraction Index_Hg_OSAR", 
    "Bout Number Index_Hg_OSAR", 
    "Bout Duration ratio_Hg_OSAR",
    "Max Velocity ratio_Hg_OSAR", 
    "Speed ratio_Hg_OSAR", 
    "Bout Speed ratio_Hg_OSAR"
]

INTENSITIES = ["Eighth", "Quarter", "Half", "Full"]

# Create mapping dict
PARAM_MAP = dict(zip(PARAMETERS, PARAMETERS_RENAME))

In [3]:
def extract_bootstrap_stats(df, metric, intensity="", metric_name=""):
    """
    Run dabest analysis and extract bootstrap statistics.
    """
    # Filter out NaN AND inf values
    df_clean = df.dropna(subset=[metric]).copy()
    df_clean = df_clean[~np.isinf(df_clean[metric])]
    
    if df_clean['status'].nunique() < 2:
        return None
    
    n_offspring = len(df_clean[df_clean['status'] == 'Offspring'])
    n_sibling = len(df_clean[df_clean['status'] == 'Sibling'])
    
    if n_offspring < 2 or n_sibling < 2:
        print(f"  [{intensity}] {metric_name}: Skipping - n_offspring={n_offspring}, n_sibling={n_sibling}")
        return None
    
    try:
        db = dabest.load(
            data=df_clean,
            x="status",
            y=metric,
            idx=("Sibling", "Offspring")
        )
        
        hedges_g_obj = db.hedges_g
        stats_results = hedges_g_obj.results
        
        # Check required columns exist
        required_cols = ['difference', 'bca_low', 'bca_high', 'bootstraps']
        missing = [c for c in required_cols if c not in stats_results.columns]
        if missing:
            print(f"  [{intensity}] {metric_name}: Missing columns {missing}")
            return None
        
        return {
            'hedges_g': stats_results['difference'].iloc[0],
            'ci_low': stats_results['bca_low'].iloc[0],
            'ci_high': stats_results['bca_high'].iloc[0],
            'bootstraps': stats_results['bootstraps'].iloc[0].tolist(),
            'n_offspring': n_offspring,
            'n_sibling': n_sibling
        }
    except Exception as e:
        print(f"  [{intensity}] {metric_name}: Error - {e}")
        return None

In [5]:
# Process and save each MBON x Responder file immediately
files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.csv')]

files = ["MB210B x ACR.csv", "SS01194 x ACR.csv", "SS75199 x ACR.csv", "SS86947 x ACR.csv", "SS97567 x ACR.csv", "SS97567 x Chrimson2.csv"]

In [6]:
for filename in files:
    base = filename.replace('.csv', '')
    parts = base.split(' x ')
    if len(parts) != 2:
        print(f"Skipping {filename} - unexpected format")
        continue
    
    mbon, responder = parts[0], parts[1]
    print(f"Processing: {mbon} x {responder}")
    
    filepath = os.path.join(INPUT_DIR, filename)
    df = pd.read_csv(filepath)
    
    # Collect results for this file
    file_results = []
    
    for intensity in INTENSITIES:
        df_intensity = df[df['light_intensity'] == intensity].copy()
        
        if len(df_intensity) == 0:
            continue
        
        for param, param_name in PARAM_MAP.items():
            if param not in df_intensity.columns:
                continue
            
            stats = extract_bootstrap_stats(df_intensity, param, intensity, param_name)
            
            if stats is None:
                continue
            
            # Create rows in long format (one row per bootstrap value)
            for bs_val in stats['bootstraps']:
                file_results.append({
                    'MBON': mbon,
                    'Responder': responder,
                    'Light_Intensity': intensity,
                    'Metric': param_name,
                    'Hedges_g': stats['hedges_g'],
                    'CI_low': stats['ci_low'],
                    'CI_high': stats['ci_high'],
                    'Bootstrap': bs_val
                })
    
    # Save this file immediately
    if file_results:
        df_out = pd.DataFrame(file_results)
        out_filename = f"{mbon} x {responder}_bootstrap.csv"
        out_path = os.path.join(OUTPUT_DIR, out_filename)
        df_out.to_csv(out_path, index=False)
        print(f"  Saved: {out_filename} ({len(df_out)} rows)")
    else:
        print(f"  No results for {mbon} x {responder}")

print("\nDone!")

Processing: MB210B x ACR
  Saved: MB210B x ACR_bootstrap.csv (180000 rows)
Processing: SS01194 x ACR
  Saved: SS01194 x ACR_bootstrap.csv (180000 rows)
Processing: SS75199 x ACR
  Saved: SS75199 x ACR_bootstrap.csv (180000 rows)
Processing: SS86947 x ACR
  Saved: SS86947 x ACR_bootstrap.csv (180000 rows)
Processing: SS97567 x ACR
  Saved: SS97567 x ACR_bootstrap.csv (180000 rows)
Processing: SS97567 x Chrimson2
  Saved: SS97567 x Chrimson2_bootstrap.csv (180000 rows)

Done!
